In [ ]:
# Install required packages (run once if not installed)
!pip install pandas numpy scikit-learn pandasql

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from pandasql import sqldf

# Set random seed for reproducibility
np.random.seed(14)

In [ ]:
# Simulated raw data (replace with your actual data loading method)
# Example: Load from CSV or database
# raw_data = pd.read_csv('your_data.csv')
# For this example, we'll assume raw_data is pre-loaded with columns: Time_Converted, TYPE, options_pct_change, equity_start_price, high_equity, low_equity, close_equity, ticker, DTE_ADJUSTED, options_earliest_open

# Sample raw data (replace with your actual data)
data = pd.DataFrame({
    'Time_Converted': pd.to_datetime(['2023-01-03 09:45:00', '2023-01-03 12:00:00', '2023-01-04 09:45:00']),
    'TYPE': ['put', 'call', 'put'],
    'options_pct_change': [10.88, 1.00, 1.42],
    'equity_start_price': [385.29, 385.29, 382.63],
    'high_equity': [385.40, 379.95, 383.99],
    'low_equity': [381.71, 379.11, 381.22],
    'close_equity': [381.71, 379.60, 383.74],
    'ticker': ['SPY', 'SPY', 'SPY'],
    'DTE_ADJUSTED': [0, 0, 0],
    'options_earliest_open': [0.3, 0.3, 0.3]
})

# Define pandasql function
pysqldf = lambda q: sqldf(q, globals())

In [ ]:
# SQL Query using pandasql
query = """
SELECT 
    o.Date,
    o.Call_Max_Daily,
    o.Put_Max_Daily,
    e.equity_open,
    e.close_equity,
    e.high_equity,
    e.low_equity
FROM (
    SELECT 
        DATE(Time_Converted) AS 'Date',
        MAX(CASE WHEN TYPE = 'call' THEN options_pct_change ELSE NULL END) AS Call_Max_Daily,
        MAX(CASE WHEN TYPE = 'put' THEN options_pct_change ELSE NULL END) AS Put_Max_Daily
    FROM data
    WHERE DTE_ADJUSTED = 0 
        AND options_earliest_open > 0.2
        AND ticker = 'SPY'
    GROUP BY DATE(Time_Converted)
) AS o
LEFT JOIN (
    SELECT
        daily_agg.Date,
        daily_agg.equity_open,
        daily_agg.high_equity,
        daily_agg.low_equity,
        last_candle.close_equity
    FROM (
        SELECT
            DATE(time_converted) AS Date,
            MAX(equity_start_price) AS equity_open,
            MAX(high_equity) AS high_equity,
            MIN(low_equity) AS low_equity
        FROM data
        WHERE ticker = 'SPY'
        GROUP BY DATE(time_converted)
    ) AS daily_agg
    LEFT JOIN (
        SELECT
            DATE(time_converted) AS Date,
            close_equity
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (PARTITION BY DATE(time_converted) ORDER BY TIME(time_converted) DESC) AS rn
            FROM data
            WHERE ticker = 'SPY'
                AND close_equity IS NOT NULL
                AND TIME(time_converted) <= '15:45:00'
        ) AS ranked_data
        WHERE rn = 1
    ) AS last_candle ON daily_agg.Date = last_candle.Date
) AS e ON o.Date = e.Date
"""

# Execute SQL query to create daily_summary
daily_summary = pysqldf(query)

# Add weekday column (simulated, adjust based on your data)
daily_summary['weekday'] = daily_summary['Date'].dt.day_name()
print(daily_summary)

In [ ]:
# Define hyperparameters and functions
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

very_high_range = 4
high_range = 2
average_range = 1.5

def day_classification(row):
    if row['Max'] > very_high_range:
        return 'very_high'
    elif row['Max'] > high_range:
        return 'high'
    elif row['Max'] > average_range:
        return 'average'
    else:
        return 'low'

# Add initial features to daily_summary
daily_summary['spy_direction_lag1'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(1)
daily_summary['intraday_range_lag1'] = (daily_summary['high_equity'] - daily_summary['low_equity']).shift(1)
daily_summary['spy_direction_lag2'] = (daily_summary['close_equity'] - daily_summary['equity_open']).shift(2)
daily_summary['Max'] = daily_summary[['Call_Max_Daily', 'Put_Max_Daily']].max(axis=1)
daily_summary['day_classification'] = daily_summary.apply(day_classification, axis=1)

def create_lags(df):
    df['d1'] = df.loc[:, 'day_classification'].shift(1)
    df['d2'] = df.loc[:, 'day_classification'].shift(2)
    df['d3'] = df.loc[:, 'day_classification'].shift(3)
    df['d4'] = df.loc[:, 'day_classification'].shift(4)
    df['d5'] = df.loc[:, 'day_classification'].shift(5)
    df['weekday_encoded'] = LabelEncoder().fit_transform(df['weekday'])
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    weekday_encoded = ohe.fit_transform(df[['weekday']])
    df = pd.concat([df, pd.DataFrame(weekday_encoded, columns=ohe.get_feature_names_out(['weekday']))], axis=1)
    df['week_number_in_month'] = df['Date'].apply(lambda x: (x.day - 1) // 7 + 1)
    df['week_sin'] = np.sin(2 * np.pi * df['week_number_in_month'] / 5)
    df['week_cos'] = np.cos(2 * np.pi * df['week_number_in_month'] / 5)
    df['max_lag1'] = df['Max'].shift(1)
    df['max_lag2'] = df['Max'].shift(2)
    df['max_lag3'] = df['Max'].shift(3)
    df['spy_direction_lag1'] = df['spy_direction_lag1']
    df['intraday_range_lag1'] = df['intraday_range_lag1']
    df['spy_direction_lag2'] = df['spy_direction_lag2']
    df['high_equity_lag1'] = df['high_equity'].shift(1)
    df['low_equity_lag1'] = df['low_equity'].shift(1)
    df.dropna(inplace=True)
    return df

def split_feature(df):
    features_list = ['d1', 'd2', 'd3', 'd4', 'd5', 'week_sin', 'week_cos', 'max_lag1', 'max_lag2', 'max_lag3', 'spy_direction_lag1', 'intraday_range_lag1', 'spy_direction_lag2']
    weekdays = [col for col in df.columns if 'weekday_' in col]
    features_list.extend(weekdays)
    X = df.loc[:, features_list]
    y = df.apply(lambda x: 1 if x['Max'] > 2 else 0, axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14, stratify=y)
    return X_train, X_test, y_train, y_test

# Create features and split data
df = create_lags(daily_summary)
X_train, X_test, y_train, y_test = split_feature(df)

# Encode categorical features
le_features = LabelEncoder()
for column in ['d1', 'd2', 'd3', 'd4', 'd5']:
    X_train[column] = le_features.fit_transform(X_train[column])
    X_test[column] = X_test[column].map(lambda s: le_features.transform([s])[0] if s in le_features.classes_ else -1)

In [ ]:
# Train and tune the volatility model (RandomForestClassifier)
rf = RandomForestClassifier(random_state=12, class_weight='balanced')
grid_search = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1, scoring='f1_weighted', verbose=2)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
model = grid_search.best_estimator_

# Evaluate the volatility model with threshold 0.55
y_pred_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_prob >= 0.55).astype(int)
print("Classification Report (Threshold 0.55):\n", classification_report(y_test, y_pred))

# Feature Importance
feature_importances = model.feature_importances_
print("Feature Importances:", dict(zip(X_train.columns, feature_importances)))

# Train directional model (RandomForestClassifier)
direction_X = df[['intraday_range_lag1', 'spy_direction_lag2', 'high_equity_lag1', 'low_equity_lag1']].dropna()
direction_y = (df['spy_direction_lag1'] > 0).astype(int).loc[direction_X.index]
print("Unique classes in direction_y:", np.unique(direction_y))
direction_X_train, direction_X_test, direction_y_train, direction_y_test = train_test_split(direction_X, direction_y, test_size=0.2, random_state=14)
direction_model = RandomForestClassifier(random_state=12, n_estimators=100)
direction_model.fit(direction_X_train, direction_y_train)
direction_pred = direction_model.predict(direction_X_test)
directional_accuracy = (direction_pred == direction_y_test).mean()
print(f"Directional Accuracy: {directional_accuracy:.2f}")

# Cross-validation
cv_scores = cross_val_score(direction_model, direction_X, direction_y, cv=5, scoring='accuracy')
print(f"Cross-validated Directional Accuracy: {cv_scores.mean():.2f} (+/- {cv_scores.std() * 2:.2f})")

# Calculate EV
test_df = X_test.copy()
test_df['y_true'] = y_test
test_df['y_pred'] = y_pred
test_df['Max'] = df.loc[X_test.index, 'Max']
wins = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])]['Max'].mean()
losses = test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] != test_df['y_true'])].shape[0]
print(f"Avg Max on wins: {wins}, Trades: {test_df['y_pred'].sum()}, Losses: {losses}")

option_cost = 0.50
tp = len(test_df[(test_df['y_pred'] == 1) & (test_df['y_pred'] == test_df['y_true'])])
fp = losses
if tp > 0:
    avg_max_wins = wins
    tp_profit = (tp * 0.5 * (option_cost * avg_max_wins - option_cost)) + (tp * 0.5 * -option_cost)
    fp_loss = fp * -option_cost
    total_profit_test = tp_profit + fp_loss
    ev_per_trade = total_profit_test / (tp + fp)
    total_trades_full = (tp + fp) * (216 / len(y_test))
    ev_per_day = (total_profit_test * (216 / len(y_test))) / 216
    print(f"EV per trade (test set): ${ev_per_trade:.2f}")
    print(f"EV per day (full dataset): ${ev_per_day:.2f}")

    # Adjust EV with directional accuracy
    if directional_accuracy > 0.5:
        adjustment_factor = directional_accuracy / 0.5
        adjusted_ev_per_day = ev_per_day * adjustment_factor
        print(f"Adjusted EV per day with {directional_accuracy:.2f} directional accuracy: ${adjusted_ev_per_day:.2f}")
